In [2]:
import pandas as pd
import os

# Configuración de rutas
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_RAW_PATH = os.path.join(BASE_DIR, "data", "raw")
FILE_NAME = "nac2018.csv" 
full_path = os.path.join(DATA_RAW_PATH, FILE_NAME)

def load_datasets(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encontró el archivo en: {path}")
    
    try:
        # AQUÍ ES DONDE SE EDITA: se agrega el parámetro encoding
        data = pd.read_csv(path, encoding='latin-1') 
        print(f"Dataset cargado exitosamente. Forma: {data.shape}")
        return data
    except Exception as e:
        print(f"Error crítico durante la lectura del archivo: {e}")
        return None

# Ejecución
df = load_datasets(full_path)

if df is not None:
    print(df.head())

Dataset cargado exitosamente. Forma: (649115, 38)
   COD_DPTO  COD_MUNIC  AREANAC  SIT_PARTO OTRO_SIT  SEXO  PESO_NAC  \
0         5          1        1          1      NaN     1         7   
1        11          1        1          1      NaN     1         6   
2        52          1        1          1      NaN     1         6   
3        11          1        1          1      NaN     1         6   
4        50          1        1          1      NaN     1         5   

   TALLA_NAC   ANO  MES  ...  AREA_RES  N_HIJOSV  FECHA_NACM  N_EMB  \
0          5  2018    1  ...       1.0         2  13/08/2014      2   
1          5  2018    1  ...       1.0         1         NaN      1   
2          5  2018    1  ...       1.0         2  15/02/2011      3   
3          5  2018    1  ...       1.0         1         NaN      2   
4          4  2018    1  ...       3.0         1         NaN      1   

   SEG_SOCIAL  IDCLASADMI  EDAD_PADRE  NIV_EDUP  ULTCURPAD  PROFESION  
0           1         1.

C:\Users\juani\AppData\Local\Temp\ipykernel_31676\3715682260.py:16: DtypeWarning: Columns (0: N_HIJOSV) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(path, encoding='latin-1')


In [3]:
# Reporte de valores faltantes
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100

missing_data = pd.DataFrame({
    'Nulos': null_counts,
    'Porcentaje (%)': null_percentages.round(2)
}).sort_values(by='Nulos', ascending=False)

print("Reporte de columnas con valores faltantes:")
print(missing_data[missing_data['Nulos'] > 0])

Reporte de columnas con valores faltantes:
             Nulos  Porcentaje (%)
OTRO_SIT    647287           99.72
FECHA_NACM  307639           47.39
IDCLASADMI   37178            5.73
CODPTORE      4091            0.63
CODMUNRE      4091            0.63
AREA_RES      3987            0.61
PROFESION        1            0.00


In [5]:
# Crear una copia para no alterar el dataframe original (Inmutabilidad)
df_cleaned = df.copy()

# A. Eliminación por irrelevancia estadística (99.7% de nulos)
# La columna 'OTRO_SIT' no aporta información útil.
df_cleaned.drop(columns=['OTRO_SIT'], inplace=True)

# B. Imputación por constante (Variables Geográficas y Administrativas)
# En lugar de borrar filas, marcamos los nulos como -1 (Desconocido).
# Esto preserva el volumen de datos para otras variables.
cols_to_fix = ['IDCLASADMI', 'CODPTORE', 'CODMUNRE', 'AREA_RES']
for col in cols_to_fix:
    df_cleaned[col] = df_cleaned[col].fillna(-1)

# C. Limpieza de registros únicos
# La columna 'PROFESION' tiene un solo nulo; procedemos a eliminar esa fila.
df_cleaned.dropna(subset=['PROFESION'], inplace=True)

print(f"Limpieza estructural completada.")

Limpieza estructural completada.


In [6]:
# Conversión forzosa a tipos numéricos
numeric_cols = ['PESO_NAC', 'TALLA_NAC', 'EDAD_PADRE', 'EDAD_MADRE']

for col in numeric_cols:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Manejo de Outliers básicos: 
# Filtramos pesos que son biológicamente imposibles para un nacido vivo (ej. > 7000g o < 400g)
# y edades de padres que podrían ser errores de captura (ej. > 100 años).
df_cleaned = df_cleaned[
    (df_cleaned['PESO_NAC'].between(400, 7000)) & 
    (df_cleaned['EDAD_PADRE'] < 100)
]

print(f"Dataset finalizado. Registros restantes: {df_cleaned.shape[0]}")

Dataset finalizado. Registros restantes: 0


In [7]:
# Definición de ruta de salida
PROCESSED_FILE = os.path.join(BASE_DIR, "data", "processed", "nac2018_cleaned.csv")

# Guardar en formato CSV estándar
df_cleaned.to_csv(PROCESSED_FILE, index=False, encoding='utf-8')
print(f"Archivo guardado exitosamente en: {PROCESSED_FILE}")

Archivo guardado exitosamente en: c:\Users\juani\OneDrive\Documents\GitHub\Proyecto-Mlops\data\processed\nac2018_cleaned.csv
